In [1]:
import numpy as np
import math
import scipy.constants as C
import warnings
from astropy import units as u

from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy.stats import sigma_clipped_stats
from astropy.wcs import WCS
from astropy.wcs.utils import proj_plane_pixel_scales

from copy import deepcopy
from glob import glob
from photutils.aperture import aperture_photometry
from photutils.aperture import SkyCircularAperture
from photutils.aperture import SkyEllipticalAperture
from regions import Regions


In [3]:
# read DS9 region file "regions_list.reg" (prepare the region file in advance)
region_list = Regions.read('ngc3351_whole.reg', format='ds9')

In [4]:
!cat ngc3351_whole.reg

# Region file format: DS9 version 4.1
global color=green dashlist=8 3 width=1 font="helvetica 10 normal roman" select=1 highlite=1 dash=0 fixed=0 edit=1 move=1 delete=1 include=1 source=1
fk5
ellipse(160.9906930,11.7037091,8.085",13.438",350)


# Write the information of the region into file with ds9 format.

In [8]:
###相比原先的版本增加了对数据内NAN的判断

def extra_region_flux(region_list, wcs, image, spec, fileout):
    """
    分析多个区域的通量和统计信息
    
    参数:
    region_list - 区域对象列表
    wcs - 图像WCS信息
    image - 2D numpy数组，表示图像数据
    spec - 用于输出文件名的标识字符串
    
    返回:
    最后一个区域的统计文件路径
    """
    # 计算像素尺度(arcsec/pixel)
    pixsc = wcs.pixel_scale_matrix.diagonal().prod()**0.5 * 3600.0
    
    # 复制图像用于背景估计
    skyimage = deepcopy(image)
    
    # 全局NaN掩码：True代表坏像素(NaN)，测光时mask=True的像素不参与积分
    nan_mask = np.isnan(image)

    # 为所有区域创建掩码并应用到背景图像
    region_masks = []
    for idx, region in enumerate(region_list):
        mask = region.to_pixel(wcs).to_mask().to_image(image.shape).astype(bool)
        region_masks.append(mask)
        skyimage[mask] = np.nan  # 排除区域像素以估计背景
    
    # 估计背景统计
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        bkg_mask = np.isnan(skyimage)
        skymean, skymed, skyrms = sigma_clipped_stats(skyimage, mask=bkg_mask, sigma=3.0)
    print(f'background statistics: mean={skymean:.5f}, median={skymed:.5f}, rms={skyrms:.5f}')
    
    # 为每个区域计算通量和统计
    output_files = []
    for idx, region in enumerate(region_list):
        # 创建孔径对象
        aperture = SkyEllipticalAperture(
            region.center,
            a=region.width/2,
            b=region.height/2,
            theta=region.angle
        )
        
        # ========= 核心改动：传入nan_mask，NaN像素直接屏蔽不参与积分 =========
        phot_table = aperture_photometry(image, aperture, wcs=wcs, mask=nan_mask)
        pixsum = phot_table['aperture_sum'][0]
        print('pixsum =', pixsum)

        # 捕获NaN兜底：如果区域内全部是坏像素，pixsum置0
        if np.isnan(pixsum):
            print(f"WARNING: region {idx} aperture_sum is NaN, set pixsum=0")
            pixsum = 0.0

        # 获取区域掩码和像素统计
        mask = region_masks[idx]
        npix = np.count_nonzero(mask)
        print(f"DEBUG region {idx}: total mask pixels npix = {npix}")

        # 合并region mask + nan_mask，传给sigma_clipped_stats
        combine_mask = np.logical_or(mask, nan_mask)
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            pixmean, pixmedian, pixstddev = sigma_clipped_stats(
                image, mask=combine_mask, sigma=3.0
            )
        
        # 提取子区域，过滤NaN
        sub_img = image[mask]
        sub_img_valid = sub_img[np.isfinite(sub_img)]
        if len(sub_img_valid) == 0:
            pixmin = 0.0
            pixmax = 0.0
            pixvar = 0.0
            print(f"WARNING: region {idx} mask内全部像素为NaN")
        else:
            pixmin = np.min(sub_img_valid)
            pixmax = np.max(sub_img_valid)
            pixvar = np.var(sub_img_valid)

        pixrms = skyrms
        # 计算孔径面积(arcsec^2)
        area_arcsec = npix * (pixsc ** 2)
        
        # 准备输出文本
        text = f"""\
center={region_list[idx].center.ra.deg} {region_list[idx].center.dec.deg}
fk5
1 pixel = {pixsc:.5g} arcsec
reg     sum             error   area            surf_bri                surf_err
                                (arcsec**2)             (sum/arcsec**2) (sum/arcsec**2)
---     ---             -----   -----------             --------------- ---------------
1       {pixsum:.8g}                ...             ...     ...             ...
reg     sum     npix    mean    median  min     max     var     stddev  rms     
---     ---     ----    ----    ------  ---     ---     ---     ------  ---     
1       {pixsum:.8g} {npix:.5g} {pixmean:.5g} {pixmedian:.5g} {pixmin:.5g} {pixmax:.5g} {pixvar:.5g} {pixstddev:.5g} {pixrms:.8g}
"""
        # 保存结果到文件
        print('spec=',spec)
        region_stats_file = f'{spec+fileout}'
        with open(region_stats_file, 'w') as fp:
            fp.write(text + '\n')
        print(f'the result output to : {region_stats_file}')
        output_files.append(region_stats_file)
    
    # 返回最后一个区域的文件路径
    return output_files[-1] if output_files else None



In [9]:
import glob
# loop images
spec_list = [
    'co10',
    'co21',
    'co32',
    'hcn10',
    'cs21',
    'hcop43',
    'cs76',
    'hcop10',
    'hcn32',
    'hcop32'
]


In [47]:
!ls *_beamsmooth_moment_0_masked.fits

M95_C3+7m_co32_co_prior_masked_cube_beamsmooth_moment_0_masked.fits
M95_C3+7m_hcop43_co_prior_masked_cube_beamsmooth_moment_0_masked.fits
M95_C5+C2_cs21_co_prior_masked_cube_beamsmooth_moment_0_masked.fits
M95_C5+C2_cs76_co_prior_masked_cube_beamsmooth_moment_0_masked.fits
M95_C5+C2_hcn10_co_prior_masked_cube_beamsmooth_moment_0_masked.fits
M95_C5+C2_hcop10_selfmask_masked_cube_beamsmooth_moment_0_masked.fits
ngc3351_12m+7m+tp_co10_selfmask_masked_cube_beamsmooth_moment_0_masked.fits
ngc3351_12m+7m+tp_co21_beamsmooth_moment_0_masked.fits
ngc3351_7m_hcn32_beamsmooth_moment_0_masked.fits
ngc3351_7m_hcop32_beamsmooth_moment_0_masked.fits


In [48]:
#file_moment_in  = '_beamsmooth_moment_0_430.fits'
#file_total_out = '_total_region_430.txt'
file_moment_in = '_beamsmooth_moment_0_masked.fits'
file_total_out = '_total_region_masked.txt'

for spec in spec_list:
    image_files = glob.glob('*'+spec+'*'+file_moment_in)
    
    if not image_files:
        print(f"No matching files found for spec: {spec}")
        continue
        
    image_file = image_files[-1]
    
    image, header = fits.getdata(image_file, header=True)
    wcs = WCS(header, naxis=2)
#    print('image_file=',image_file)
    spec_region= extra_region_flux(region_list,wcs,image,spec, file_total_out)
    print('spec_region=',spec_region)

/Users/hjma/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:17: RuntimeWarning: invalid value encountered in double_scalars
/Users/hjma/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:17: RuntimeWarning: invalid value encountered in double_scalars
/Users/hjma/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:17: RuntimeWarning: invalid value encountered in double_scalars


background statistics: mean=31.55629, median=25.66763, rms=21.76489
pixsum = 2390740.9539434165
DEBUG region 0: total mask pixels npix = 34138
spec= co10
the result output to : co10_total_region_masked.txt
spec_region= co10_total_region_masked.txt
background statistics: mean=3.43666, median=3.10297, rms=7.79330
pixsum = 2207202.4227560754
DEBUG region 0: total mask pixels npix = 34138
spec= co21
the result output to : co21_total_region_masked.txt
spec_region= co21_total_region_masked.txt
background statistics: mean=1.76537, median=1.19895, rms=4.14326
pixsum = 1520495.478662636
DEBUG region 0: total mask pixels npix = 34138
spec= co32
the result output to : co32_total_region_masked.txt
spec_region= co32_total_region_masked.txt


/Users/hjma/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:17: RuntimeWarning: invalid value encountered in double_scalars
/Users/hjma/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:17: RuntimeWarning: invalid value encountered in double_scalars
/Users/hjma/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:17: RuntimeWarning: invalid value encountered in double_scalars
/Users/hjma/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:17: RuntimeWarning: invalid value encountered in double_scalars


background statistics: mean=4.26047, median=3.94982, rms=2.10977
pixsum = 293209.4336312857
DEBUG region 0: total mask pixels npix = 34138
spec= hcn10
the result output to : hcn10_total_region_masked.txt
spec_region= hcn10_total_region_masked.txt
background statistics: mean=5.17195, median=5.18890, rms=1.60888
pixsum = 136645.90404232088
DEBUG region 0: total mask pixels npix = 34138
spec= cs21
the result output to : cs21_total_region_masked.txt
spec_region= cs21_total_region_masked.txt
background statistics: mean=1.50653, median=1.39859, rms=0.53519
pixsum = 29031.671796660936
DEBUG region 0: total mask pixels npix = 34138
spec= hcop43
the result output to : hcop43_total_region_masked.txt
spec_region= hcop43_total_region_masked.txt
background statistics: mean=1.24556, median=1.24408, rms=0.46199
pixsum = 11156.731513877588
DEBUG region 0: total mask pixels npix = 34138
spec= cs76
the result output to : cs76_total_region_masked.txt
spec_region= cs76_total_region_masked.txt


/Users/hjma/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:17: RuntimeWarning: invalid value encountered in double_scalars
/Users/hjma/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:17: RuntimeWarning: invalid value encountered in double_scalars
/Users/hjma/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:17: RuntimeWarning: invalid value encountered in double_scalars


background statistics: mean=5.51149, median=5.58877, rms=1.84601
pixsum = 219580.92988848028
DEBUG region 0: total mask pixels npix = 34138
spec= hcop10
the result output to : hcop10_total_region_masked.txt
spec_region= hcop10_total_region_masked.txt
background statistics: mean=-0.02199, median=-0.06182, rms=0.49025
pixsum = 20217.43550730646
DEBUG region 0: total mask pixels npix = 34138
spec= hcn32
the result output to : hcn32_total_region_masked.txt
spec_region= hcn32_total_region_masked.txt
background statistics: mean=-0.08220, median=-0.08204, rms=0.35150
pixsum = 19059.211043448777
DEBUG region 0: total mask pixels npix = 34138
spec= hcop32
the result output to : hcop32_total_region_masked.txt
spec_region= hcop32_total_region_masked.txt


In [11]:
!ls *_total_region_masked.txt

co10_total_region_masked.txt   hcn32_total_region_masked.txt
co32_total_region_masked.txt   hcop10_total_region_masked.txt
cs21_total_region_masked.txt   hcop32_total_region_masked.txt
cs76_total_region_masked.txt   hcop43_total_region_masked.txt
hcn10_total_region_masked.txt


In [49]:
!cat cs21_total_region_masked.txt

center=160.990693 11.7037091
fk5
1 pixel = nan arcsec
reg     sum             error   area            surf_bri                surf_err
                                (arcsec**2)             (sum/arcsec**2) (sum/arcsec**2)
---     ---             -----   -----------             --------------- ---------------
1       2207202.4                ...             ...     ...             ...
reg     sum     npix    mean    median  min     max     var     stddev  rms     
---     ---     ----    ----    ------  ---     ---     ---     ------  ---     
1       2207202.4 34138 3.4367 3.103 -15.556 335.13 5773.5 7.7933 7.7932978



In [7]:
!cat cs76_total_region_masked.txt

center=160.990693 11.7037091
fk5
1 pixel = nan arcsec
reg     sum             error   area            surf_bri                surf_err
                                (arcsec**2)             (sum/arcsec**2) (sum/arcsec**2)
---     ---             -----   -----------             --------------- ---------------
1       11156.732                ...             ...     ...             ...
reg     sum     npix    mean    median  min     max     var     stddev  rms     
---     ---     ----    ----    ------  ---     ---     ---     ------  ---     
1       11156.732 34138 1.2456 1.2441 0.24187 3.8056 0.3079 0.46199 0.46198918



## translate K km/s into Jy km/s 

In [13]:
def read_txt(file):
    with open(file, "r",encoding='utf-8') as f:
        lines = f.readlines()
        return lines

In [50]:
import glob
## read the frequence and beam of line
def read_fre_beam(file_moment_in, spec):
    fi_out     = glob.glob(f"*_{spec}*{file_moment_in}")#'*'+spec+ filein)
    moment_hdu = fits.open(fi_out[0])
    print(moment_hdu)

    freq_hz  = moment_hdu[0].header['RESTFRQ']   #Hz
    newmajor = float(moment_hdu[0].header['BEAM'].split()[1].strip('BMAJ='))   #arcsec
    newminor = float(moment_hdu[0].header['BEAM'].split()[3].strip('BMIN='))  #arcsec
    bmaj_as  = newmajor
    bmin_as  = newminor
    print('freq_hz, bmaj_as, bmin_as=',freq_hz, bmaj_as, bmin_as)
    return freq_hz, bmaj_as, bmin_as


In [51]:
def jtok(freq_hz, bmaj_as, bmin_as):
    """
    convert the Kelven to Jyansky
    
    参数:
    freq_hz - The frequence of spectrum
    bmaj_as - the beam major axis 
    bmin_as - the beam minor axis 
    
    return:
    the flux with Jyansky
    """
    c = 2.99792458e10
    h = 6.6260755e-27
    kb = 1.380658e-16
    bmaj_sr = (bmaj_as*u.arcsec).to(u.rad).value # bmaj_as/3600.*np.pi/180.
    bmin_sr = (bmin_as*u.arcsec).to(u.rad).value # bmin_as/3600.*np.pi/180.
    #beam_in_as = np.pi*(bmaj_as/2.0*bmin_as/2.0)/np.log(2)
    
    beam_in_sr = np.pi*(bmaj_sr/2.0*bmin_sr/2.0)/np.log(2)
    
    Jy_to_K_values = c**2 / beam_in_sr / 1e23 / (2*kb*freq_hz**2)
    jtok_value = np.mean(Jy_to_K_values)
    
    return jtok_value

In [52]:
!ls ngc3351_12m+7m+tp_co21.fits

ngc3351_12m+7m+tp_co21.fits


In [5]:
!ls *masked_cube.fits

M95_C3+7m_co32_co_prior_masked_cube.fits
M95_C3+7m_hcop43_co_prior_masked_cube.fits
M95_C5+C2_cs21_co_prior_masked_cube.fits
M95_C5+C2_cs76_co_prior_masked_cube.fits
M95_C5+C2_hcn10_co_prior_masked_cube.fits
M95_C5+C2_hcop10_selfmask_masked_cube.fits
ngc3351_12m+7m+tp_co10_selfmask_masked_cube.fits


In [53]:
# the cube files 
cube_dict = {}
cube_dict['co21'] = 'ngc3351_12m+7m+tp_co21.fits'
cube_dict['co32'] = 'M95_C3+7m_co32_co_prior_masked_cube.fits'
cube_dict['hcn10'] = 'M95_C5+C2_hcn10_co_prior_masked_cube.fits'
cube_dict['cs21'] = 'M95_C5+C2_cs21_co_prior_masked_cube.fits'
cube_dict['hcop43'] = 'M95_C3+7m_hcop43_co_prior_masked_cube.fits'
cube_dict['cs76'] = 'M95_C5+C2_cs76_co_prior_masked_cube.fits'
cube_dict['hcop10'] = 'M95_C5+C2_hcop10_selfmask_masked_cube.fits'
cube_dict['co10'] = 'ngc3351_12m+7m+tp_co10_selfmask_masked_cube.fits'
cube_dict['hcn32'] = 'ngc3351_7m_hcn32_moment_0.fits'
cube_dict['hcop32'] = 'ngc3351_7m_hcop32_moment_0.fits'

In [54]:
# the rms found by hand
rms_dict = {}
rms_dict['co32'] = 4.00201
rms_dict['hcn10'] = 4.68409
rms_dict['cs21'] = 3.49708
rms_dict['hcop43'] = 1.772
rms_dict['cs76'] = 1.04599
rms_dict['hcop10'] = 4.18591
rms_dict['co21'] = 10.7978
rms_dict['co10'] = 0.01
rms_dict['hcn32'] = 0.01
rms_dict['hcop32'] = 0.01


In [55]:

def read_spec_info(spec, filine, freq_hz, bmaj_as, bmin_as, f_flux_out):
    
    """
    读取并处理谱线信息，计算通量和误差
    
    参数:
    spec - 谱线标识
    filine - 输入文件名
    freq_hz - 频率(Hz)
    bmaj_as - 主波束尺寸(arcsec)
    bmin_as - 次波束尺寸(arcsec)
    f_flux - 输出文件对象
    
    返回:
    None
    """
    # 读取并解析文件
    fil = filine.strip('\n')
    lins = read_txt(fil)

    # 解析中心坐标
    cen = lins[0]
    cen = cen.split('center=')[1].replace('/r','').strip()
    Ra,Dec =cen.split()
    
    # 解析积分强度
    data_line = lins[7]
    inten_int = data_line.split()[1]
    
    # 检查Jansky-to-Kelvin转换因子
    print('Checking JTOK: {} {} (freq={}, bmaj={}, bmin={})'.format(
        jtok(freq_hz, bmaj_as, bmin_as), 
        JYTOK_dict[spec],
        freq_hz, bmaj_as, bmin_as, 
    ))
    jtokvalue = jtok(freq_hz, bmaj_as, bmin_as)
       
    ## the pixel size 
    #if freq_hz == 267557630000.0 or freq_hz == 265886180000.0:
    #    pixsize = 0.0005555555555556 * 3600.0 # arcsec #for HCO+(3-2) and HCN(3-2)
        
    #else:
        #pixsize = 2.777777777778E-05 * 3600.0 # arcsec #<TODO>#
   
    pixsize = 2.777777777778E-05 * 3600.0 # arcsec #<TODO>#
    
    # 计算波束与像素面积比
    beampixratio = np.pi/(4.*np.log(2))*bmaj_as*bmin_as/(pixsize**2)    
    
    # 计算通量(Jy)
    flux_Jy = float(inten_int)/(jtokvalue*beampixratio)
    ##flux_Jy = float(inten_int)/JYTOK_dict[spec]
    flux_Jy = str(flux_Jy)
     
    # 解析RMS和像素数
    std_line = lins[11]
    pixrms = float(std_line.split()[-1])
    npix = int(std_line.split()[2])
   
    # 检查RMS
    print('Checking RMS: {} {} (photutil rms vs. rms dict)'.format(
        pixrms,
        rms_dict[spec], 
    ))
    
    # 计算误差
    stddev   = str(pixrms * np.sqrt(float(npix)) )
    err_flux_Jy = float(stddev)/(jtokvalue*beampixratio)
    ##err_flux_Jy = float(stddev)/JYTOK_dict[spec]
    err_flux_Jy = str(err_flux_Jy)
    
    ##stddev   = str(rms_dict[spec] * np.sqrt(float(std_line.split()[2])) )
    ####stddev   = str(pixrms * np.sqrt(float(npix)) )
    ####err_flux_Jy = float(stddev)/jtok(freq_hz, bmaj_as, bmin_as)
    ##err_flux_Jy = float(stddev)/JYTOK_dict[spec]
    ####err_flux_Jy = str(err_flux_Jy)
    
    # 计算波长(um)
    wav_len = C.c/freq_hz*1e6 #um
    wavelen = str(wav_len)
     
    # 写入输出文件
#    print(wavelen,Ra, Dec, inten_int, flux_Jy, stddev, err_flux_Jy)
    f_flux_out.write(spec+'     '+wavelen+'   '+ Ra + '  ' + Dec +'   '+inten_int + '  '+flux_Jy +'   '+stddev+'   '+err_flux_Jy+'\n') 
       
    return



In [56]:
def read_spec_info(spec, filine, freq_hz, bmaj_as, bmin_as, f_flux_out):
    
    """
    读取并处理谱线信息，计算通量和误差
    
    参数:
    spec - 谱线标识
    filine - 输入文件名
    freq_hz - 频率(Hz)
    bmaj_as - 主波束尺寸(arcsec)
    bmin_as - 次波束尺寸(arcsec)
    f_flux - 输出文件对象
    
    返回:
    None
    """
    # 读取并解析文件
    fil = filine.strip('\n')
    lins = read_txt(fil)
    
    # 解析中心坐标（第一行保留不变）
    cen = lins[0]
    cen = cen.split('center=')[1].replace('/r','').strip()
    Ra,Dec =cen.split()
    
    # ========== 【修改这里：自动找imstat数据行，不再硬编码lins[7]/lins[11]】 ==========
    data_line = None
    for line in lins:
        parts = line.strip().split()
        if len(parts) < 10:
            continue
        try:
            float(parts[0])
            float(parts[1]) # sum
            float(parts[2]) # npix
            float(parts[-1])# rms
            data_line = parts
            break
        except ValueError:
            continue
    if data_line is None:
        raise RuntimeError(f"文件 {fil} 找不到imstat数字数据行！")

    inten_int = data_line[1]   # sum 积分值
    npix = int(data_line[2])
    pixrms = float(data_line[-1])
    # ==================================================================================
    
    # 检查Jansky-to-Kelvin转换因子
    print('Checking JTOK: {} {} (freq={}, bmaj={}, bmin={})'.format(
        jtok(freq_hz, bmaj_as, bmin_as), 
        JYTOK_dict[spec],
        freq_hz, bmaj_as, bmin_as, 
    ))
    jtokvalue = jtok(freq_hz, bmaj_as, bmin_as)
       
    pixsize = 2.777777777778E-05 * 3600.0 # arcsec 
    
    # 计算波束与像素面积比
    beampixratio = np.pi/(4.*np.log(2))*bmaj_as*bmin_as/(pixsize**2)    
    
    # 计算通量(Jy)
    flux_Jy = float(inten_int)/(jtokvalue*beampixratio)
    flux_Jy = str(flux_Jy)
     
    # 计算误差
    stddev = pixrms * np.sqrt(float(npix))
    err_flux_Jy = stddev/(jtokvalue*beampixratio)
    err_flux_Jy = str(err_flux_Jy)
    
    # 计算波长(um)
    wav_len = C.c/freq_hz*1e6 #um
    wavelen = str(wav_len)
     
    # 写入输出文件
    f_flux_out.write(spec+'     '+wavelen+'   '+ Ra + '  ' + Dec +'   '+inten_int + '  '+flux_Jy +'   '+str(stddev)+'   '+err_flux_Jy+'\n') 
       
    return

In [39]:
#read JytoK from orignal fits file. Comparing the result of Jytok Vs JYTOK.
JYTOK_dict ={}
JYTOK_dict['co21']=10.79531919804
JYTOK_dict['co32']=31.20437007042
JYTOK_dict['hcn10']=188.2359894961
JYTOK_dict['cs21']=177.1387871622
JYTOK_dict['hcop43']=19.83662890417
JYTOK_dict['cs76']=20.08020409516
JYTOK_dict['hcop10']=187.9056062173
JYTOK_dict['co10']=0.01
JYTOK_dict['hcn32']=0.01
JYTOK_dict['hcop32']=4.749561628750E-01

In [44]:
!ls *co21*.txt

co21_total_flux_summary_masked.txt


In [8]:
!ls *_beamsmooth_moment_0_masked.fits

M95_C3+7m_co32_co_prior_masked_cube_beamsmooth_moment_0_masked.fits
M95_C3+7m_hcop43_co_prior_masked_cube_beamsmooth_moment_0_masked.fits
M95_C5+C2_cs21_co_prior_masked_cube_beamsmooth_moment_0_masked.fits
M95_C5+C2_cs76_co_prior_masked_cube_beamsmooth_moment_0_masked.fits
M95_C5+C2_hcn10_co_prior_masked_cube_beamsmooth_moment_0_masked.fits
M95_C5+C2_hcop10_selfmask_masked_cube_beamsmooth_moment_0_masked.fits
ngc3351_12m+7m+tp_co10_selfmask_masked_cube_beamsmooth_moment_0_masked.fits
ngc3351_12m+7m+tp_co21_beamsmooth_moment_0_masked.fits
ngc3351_7m_hcn32_beamsmooth_moment_0_masked.fits
ngc3351_7m_hcop32_beamsmooth_moment_0_masked.fits


In [57]:
#file_moment_in ='_moment_0_430.fits'
#file_total_out = '_total_region_430.txt'
#file_total_summary_out = '_total_flux_summary_430.txt'
file_moment_in = '_beamsmooth_moment_0_masked.fits'
file_total_out = '_total_region_masked.txt'

file_total_summary_out = '_total_flux_summary_masked.txt'

for ispec, spec in enumerate(spec_list):
    
    # 使用 with 语句打开文件，确保文件在使用完后自动关闭
    with open(f"{spec+file_total_summary_out}", 'w+') as f_flux_out:
        # 使用 f-string 进行字符串拼接，提高可读性
        f_flux_out.write(f"spec    wavleng_um    RA.     DEC     {spec}integrade(K_km/s)  {spec}flux(Jy_km/s)  {spec}stddev(K_km/s)  {spec}err_flux(Jy_km/s)\n")
        
        for ireg in range(len(region_list)):
            region_stats_file = f'{spec+file_total_out}'
            try:
                # 调用 read_fre_beam 函数获取频率和波束信息
                freq_hz, bmaj_as, bmin_as = read_fre_beam(file_moment_in, spec)
                print(f'freq_hz, bmaj_as, bmin_as= {freq_hz}, {bmaj_as}, {bmin_as}')
                
                # 调用 read_spec_info 函数处理谱线信息
                read_spec_info(spec, region_stats_file, freq_hz, bmaj_as, bmin_as, f_flux_out)
                print(region_stats_file)
            
            except Exception as e:
                print(f"处理 {region_stats_file} 时出现错误: {e}")

freq_hz, bmaj_as, bmin_as= 115271200000.0 1.9593611996868001 1.9593611996868001
freq_hz, bmaj_as, bmin_as= 115271200000.0, 1.9593611996868001, 1.9593611996868001
Checking JTOK: 23.9574453762374 0.01 (freq=115271200000.0, bmaj=1.9593611996868001, bmin=1.9593611996868001)
co10_total_region_masked.txt
freq_hz, bmaj_as, bmin_as= 230538000000.0 1.9593611996868001 1.9593611996868001
freq_hz, bmaj_as, bmin_as= 230538000000.0, 1.9593611996868001, 1.9593611996868001
Checking JTOK: 5.989589969634305 10.79531919804 (freq=230538000000.0, bmaj=1.9593611996868001, bmin=1.9593611996868001)
co21_total_region_masked.txt
freq_hz, bmaj_as, bmin_as= 345795990000.0 1.9593611996868001 1.9593611996868001
freq_hz, bmaj_as, bmin_as= 345795990000.0, 1.9593611996868001, 1.9593611996868001
Checking JTOK: 2.6622095056870823 31.20437007042 (freq=345795990000.0, bmaj=1.9593611996868001, bmin=1.9593611996868001)
co32_total_region_masked.txt
freq_hz, bmaj_as, bmin_as= 88631850000.0 1.9593611996868001 1.959361199686800

In [4]:
!ls *_total_flux_summary_masked.txt
!cat cs76_total_flux_summary_masked.txt

co10_total_flux_summary_masked.txt   hcn10_total_flux_summary_masked.txt
co21_total_flux_summary_masked.txt   hcn32_total_flux_summary_masked.txt
co32_total_flux_summary_masked.txt   hcop10_total_flux_summary_masked.txt
cs21_total_flux_summary_masked.txt   hcop32_total_flux_summary_masked.txt
cs76_total_flux_summary_masked.txt   hcop43_total_flux_summary_masked.txt
spec    wavleng_um    RA.     DEC     cs76integrade(K_km/s)  cs76flux(Jy_km/s)  cs76stddev(K_km/s)  cs76err_flux(Jy_km/s)
cs76     874.3291126984042   160.990693  11.7037091   11156.732  9.472245926182671   85.35929928951371   0.07247142576848702


In [9]:
!cat cs21_total_flux_summary_masked.txt

spec    wavleng_um    RA.     DEC     cs21integrade(K_km/s)  cs21flux(Jy_km/s)  cs21stddev(K_km/s)  cs21err_flux(Jy_km/s)
cs21     3059.701482788236   160.990693  11.7037091   136645.9  9.473366393402612   297.26501260059797   0.020608744062609336


In [83]:
!ls ../*total_flux_summary_430.txt
!cat ../hcop10_total_flux_summary_430.txt

../co10_total_flux_summary_430.txt   ../hcn10_total_flux_summary_430.txt
../co21_total_flux_summary_430.txt   ../hcn32_total_flux_summary_430.txt
../co32_total_flux_summary_430.txt   ../hcop10_total_flux_summary_430.txt
../cs21_total_flux_summary_430.txt   ../hcop32_total_flux_summary_430.txt
../cs76_total_flux_summary_430.txt   ../hcop43_total_flux_summary_430.txt
spec    wavleng_um    RA.     DEC     hcop10integrade(K_km/s)  hcop10flux(Jy_km/s)  hcop10stddev(K_km/s)  hcop10err_flux(Jy_km/s)
hcop10     3361.3345977711033   160.990693  11.7037091   126803.66  7.284070502700704   762.6837828231791   0.043811372994679396


# write the flux and std value of different spectrum into a file

In [143]:
!paste *_total_flux_summary_430.txt >summary_specs_total_flux_430.txt
#!cat summary_specs_total_flux.txt

In [144]:
!cat summary_specs_total_flux_430.txt

spec    wavleng_um    RA.     DEC     co10integrade(K_km/s)  co10flux(Jy_km/s)  co10stddev(K_km/s)  co10err_flux(Jy_km/s)	spec    wavleng_um    RA.     DEC     co21integrade(K_km/s)  co21flux(Jy_km/s)  co21stddev(K_km/s)  co21err_flux(Jy_km/s)	spec    wavleng_um    RA.     DEC     co32integrade(K_km/s)  co32flux(Jy_km/s)  co32stddev(K_km/s)  co32err_flux(Jy_km/s)	spec    wavleng_um    RA.     DEC     cs21integrade(K_km/s)  cs21flux(Jy_km/s)  cs21stddev(K_km/s)  cs21err_flux(Jy_km/s)	spec    wavleng_um    RA.     DEC     cs76integrade(K_km/s)  cs76flux(Jy_km/s)  cs76stddev(K_km/s)  cs76err_flux(Jy_km/s)	spec    wavleng_um    RA.     DEC     hcn10integrade(K_km/s)  hcn10flux(Jy_km/s)  hcn10stddev(K_km/s)  hcn10err_flux(Jy_km/s)	spec    wavleng_um    RA.     DEC     hcn32integrade(K_km/s)  hcn32flux(Jy_km/s)  hcn32stddev(K_km/s)  hcn32err_flux(Jy_km/s)	spec    wavleng_um    RA.     DEC     hcop10integrade(K_km/s)  hcop10flux(Jy_km/s)  hcop10stddev(K_km/s)  hcop10err_flux(Jy_km/s)	spec    

# write into sed and lvg file

In [35]:
def sed_lvgdatawrite(line, file_out1, file_out2):
    num = int(len(line.split())/8)
    for i in range(0, num):
        speclin  = line.split()[i*8+0]
        wavelen  = line.split()[i*8+1]
        flux     = line.split()[i*8+5]
        flux_err = line.split()[i*8+7]
        
#        print(wavelen, flux, flux_err)
        file_out1.write(wavelen+'  '+flux+'  '+flux_err+'\n')
        file_out2.write(speclin+'  '+flux+'  '+flux_err+'\n')
        
    return 

In [36]:
def lvg_datawrite(fi1, fi2, fi_out):
    lines_fi1 = read_txt(fi1)
    lines_fi2 = read_txt(fi2)
    for i in range(1,len(lines_fi1)):
        spec_fi1, flux, flux_err= lines_fi1[i].split()
        
        for k in range(0,len(lines_fi2)):
            spec_fi2, x_spec = lines_fi2[k].split()
            
            if spec_fi2==spec_fi1:
#                print(spec_fi1,flux,spec_fi2,x_spec)

                if str(x_spec)[0:3] == '101': # for CO line, multiplying a weighting factor
                    weightCO = 0.01 # make sure this is the same as in "makeliblvg_z_with_CO_HCN_CS_HCOP.sm"
                    flux = str(float(flux)*weightCO)
                    flux_err = str(float(flux_err)*weightCO)
                    weight = str(weightCO)
                else:
                    weight = str(1.0)
                    
                fi_out.write(x_spec+'   '+flux+'    '+flux_err+'     '+weight+'     '+spec_fi1+'\n')
    return
        

In [37]:
filout_spec = open('spectral_species.txt', 'w+')

filout_spec.write('line'+'    X_species'+ '\n')
filout_spec.write('co10'+'    101001000'+ '\n')
filout_spec.write('co21'+'    101002001'+ '\n')
filout_spec.write('co32'+'    101003002'+ '\n')
filout_spec.write('hcn10'+'   102001000'+ '\n')
filout_spec.write('hcn32'+'   102003002'+ '\n')
filout_spec.write('cs21'+'    103002001'+ '\n')
filout_spec.write('cs76'+'    103007006'+ '\n')
filout_spec.write('hcop10'+'  104001000'+ '\n')
filout_spec.write('hcop32'+'  104003002'+ '\n')
filout_spec.write('hcop43'+'  104004003'+ '\n')
filout_spec.close()

In [38]:
!cat spectral_species.txt

line    X_species
co10    101001000
co21    101002001
co32    101003002
hcn10   102001000
hcn32   102003002
cs21    103002001
cs76    103007006
hcop10  104001000
hcop32  104003002
hcop43  104004003


In [148]:
#ile_in = 'summary_specs_total_flux.txt'
file_in = 'summary_specs_total_flux_430.txt'
li_data = read_txt(file_in)
lines_num = len(li_data)

fi2 = 'spectral_species.txt'

for j in range(1,lines_num):

    ##### for data of SED fitting  
    file_out_sed = open('sed_flux_region'+str(j)+'total_430.txt','w+')
    file_out_sed.write("#wavelenght_um"+ '     flux_Jy_km/s'+'     err_flux_Jy_km/s' +'\n')
    
    file_name = 'spec_region'+str(j)+'_total_430.txt'
    file_out  = open(file_name,'w+')
    file_out.write("#sepc"+'       flux_Jy_km/s'+'       err_flux_Jy_km/s' +'\n')
    
    line= li_data[j]
    sed_lvgdatawrite(line, file_out_sed,file_out)
    file_out.close()
    
    ##### for data of LVG fitting  
    fi1 = file_name
    fi_out = open('lvg_data_total_region'+str(j)+'_beam_pixel_total_430.txt','w+')
    fi_out.write('#x_speceies'+'   flux_Jy_km/s'+'   err_flux_Jy_km/s'+ '       weight'+'        spec'+'\n')   
    lvg_datawrite(fi1, fi2, fi_out)

    
#file_out_sed.close()
fi_out.close()

In [149]:
!ls *total_430.txt

lvg_data_total_region1_beam_pixel_total_430.txt
sed_flux_region1total_430.txt
spec_region1_total_430.txt


In [28]:
!ls *spec_region*.txt

spec_region0.txt           spec_region2.txt
spec_region1.txt           spec_region3.txt
spec_region10.txt          spec_region4.txt
spec_region11.txt          spec_region5.txt
spec_region12.txt          spec_region6.txt
spec_region13.txt          spec_region7.txt
spec_region1_total.txt     spec_region8.txt
spec_region1_total_428.txt spec_region9.txt
spec_region1_total_430.txt


In [29]:
!cat spec_region1_total_430.txt

#sepc       flux_Jy_km/s       err_flux_Jy_km/s
co10  235.99928401389633  0.22337981589223277
co21  847.1330765219033  0.5526495089783635
co32  1344.1308963843978  0.8946120261791126
cs21  5.955802684781036  0.046363899369995944
cs76  1.8278144903618994  0.21560689801144547
hcn10  12.451845493170538  0.04588593916206363
hcn32  10.32148161244036  0.04624396264631994
hcop10  7.284070502700704  0.043811372994679396
hcop32  9.852899112683666  0.033574237095961086
hcop43  11.684727658500394  0.24748632385564814


In [30]:
!cat spec_region1_total_428.txt

#sepc       flux_Jy_km/s       err_flux_Jy_km/s
co10  235.99928401389633  0.22337981589223277
co21  847.1330765219033  0.5526495089783635
co32  1344.1308963843978  0.8946120261791126
cs21  5.955802684781036  0.046363899369995944
cs76  1.8278144903618994  0.21560689801144547
hcn10  12.451845493170538  0.04588593916206363
hcn32  10.32148161244036  0.04624396264631994
hcop10  7.284070502700704  0.043811372994679396
hcop32  9.852899112683666  0.033574237095961086
hcop43  11.684727658500394  0.24748632385564814


-0.7985076962177716

In [8]:
2.003852343742E-04*3600

0.72138684374712